1. Prepare the dataset using Pandas

In [1]:
import pandas as pd

# --------------------------------------------------
# 1. CREATE THE DATASET
# --------------------------------------------------

data = [
    {
        "Timestamp": "09:00",
        "User": "buyer01",
        "Event": "Login",
        "Source IP": "10.0.0.12",
        "Result": "Success",
    },
    {
        "Timestamp": "09:02",
        "User": "buyer01",
        "Event": "Login",
        "Source IP": "203.0.113.25",
        "Result": "Failure",
    },
    {
        "Timestamp": "09:03",
        "User": "buyer01",
        "Event": "Login",
        "Source IP": "203.0.113.25",
        "Result": "Failure",
    },
    {
        "Timestamp": "09:04",
        "User": "buyer01",
        "Event": "Login",
        "Source IP": "203.0.113.25",
        "Result": "Success",
    },
    {
        "Timestamp": "09:07",
        "User": "buyer01",
        "Event": "Supplier record viewed",
        "Source IP": "203.0.113.25",
        "Result": "Success",
    },
    {
        "Timestamp": "09:10",
        "User": "buyer01",
        "Event": "Bank details modified",
        "Source IP": "203.0.113.25",
        "Result": "Success",
    },
]

df = pd.DataFrame(data)

# Convert timestamps to datetime objects.
# The date is illustrative because the original dataset
# provides times but no calendar date or timezone.
df["Timestamp"] = pd.to_datetime(
    "2026-10-09 " + df["Timestamp"],
    format="%Y-%m-%d %H:%M",
)

# Ensure chronological ordering.
df = df.sort_values("Timestamp").reset_index(drop=True)

print("RAW SECURITY LOGS")
print(df.to_string(index=False))

RAW SECURITY LOGS
          Timestamp    User                  Event    Source IP  Result
2026-10-09 09:00:00 buyer01                  Login    10.0.0.12 Success
2026-10-09 09:02:00 buyer01                  Login 203.0.113.25 Failure
2026-10-09 09:03:00 buyer01                  Login 203.0.113.25 Failure
2026-10-09 09:04:00 buyer01                  Login 203.0.113.25 Success
2026-10-09 09:07:00 buyer01 Supplier record viewed 203.0.113.25 Success
2026-10-09 09:10:00 buyer01  Bank details modified 203.0.113.25 Success


2. Calculate failed logins per user

In [2]:
login_events = df[
    df["Event"].eq("Login")
].copy()

failed_logins = (
    login_events[login_events["Result"].eq("Failure")]
    .groupby("User")
    .size()
    .rename("Failed Logins")
)

successful_logins = (
    login_events[login_events["Result"].eq("Success")]
    .groupby("User")
    .size()
    .rename("Successful Logins")
)

login_summary = pd.concat(
    [failed_logins, successful_logins],
    axis=1,
).fillna(0).astype(int)

print("\nLOGIN SUMMARY")
print(login_summary)


LOGIN SUMMARY
         Failed Logins  Successful Logins
User                                     
buyer01              2                  2


3. Calculate the failed-login rate

The formula is:

Failed-login rate= Total login attempts / Failed login attempts × 100 
	​

In [3]:
total_attempts = len(login_events)

failed_attempts = (
    login_events["Result"].eq("Failure").sum()
)

failed_login_rate = (
    failed_attempts / total_attempts * 100
    if total_attempts > 0
    else 0
)

print(f"Total login attempts: {total_attempts}")
print(f"Failed login attempts: {failed_attempts}")
print(f"Failed-login rate: {failed_login_rate:.2f}%")

Total login attempts: 4
Failed login attempts: 2
Failed-login rate: 50.00%


4. Identify successful logins after repeated failures

In [4]:
login_events = login_events.sort_values("Timestamp")

suspicious_logins = []

for (user, source_ip), group in login_events.groupby(
    ["User", "Source IP"]
):
    failed_count = 0

    for _, row in group.iterrows():

        if row["Result"] == "Failure":
            failed_count += 1

        elif row["Result"] == "Success":

            if failed_count >= 2:
                suspicious_logins.append({
                    "User": user,
                    "Source IP": source_ip,
                    "Successful Login": row["Timestamp"],
                    "Prior Failed Logins": failed_count,
                })

            failed_count = 0

suspicious_login_df = pd.DataFrame(suspicious_logins)

print("\nSUCCESSFUL LOGINS AFTER REPEATED FAILURES")
print(suspicious_login_df.to_string(index=False))


SUCCESSFUL LOGINS AFTER REPEATED FAILURES
   User    Source IP    Successful Login  Prior Failed Logins
buyer01 203.0.113.25 2026-10-09 09:04:00                    2


5. Detect changes to sensitive supplier records

In [5]:
sensitive_keywords = [
    "bank details modified",
    "supplier record modified",
    "payment details modified",
    "supplier bank account changed",
]

sensitive_mask = (
    df["Event"]
    .str.lower()
    .isin(sensitive_keywords)
)

sensitive_changes = df[sensitive_mask].copy()

print("\nSENSITIVE SUPPLIER CHANGES")
print(
    sensitive_changes[
        ["Timestamp", "User", "Event", "Source IP", "Result"]
    ].to_string(index=False)
)


SENSITIVE SUPPLIER CHANGES
          Timestamp    User                 Event    Source IP  Result
2026-10-09 09:10:00 buyer01 Bank details modified 203.0.113.25 Success


6. Calculate the time between authentication and sensitive activity

In [6]:
successful_auth = df[
    (df["Event"] == "Login")
    & (df["Result"] == "Success")
]

sensitive_events = df[
    df["Event"].str.lower().isin(sensitive_keywords)
]

time_differences = []

for _, activity in sensitive_events.iterrows():

    prior_auth = successful_auth[
        (successful_auth["User"] == activity["User"])
        & (
            successful_auth["Source IP"]
            == activity["Source IP"]
        )
        & (
            successful_auth["Timestamp"]
            <= activity["Timestamp"]
        )
    ]

    if not prior_auth.empty:
        latest_auth = prior_auth.iloc[-1]

        elapsed = (
            activity["Timestamp"] - latest_auth["Timestamp"]
        )

        time_differences.append({
            "User": activity["User"],
            "Source IP": activity["Source IP"],
            "Authentication Time": latest_auth["Timestamp"],
            "Sensitive Activity": activity["Event"],
            "Activity Time": activity["Timestamp"],
            "Elapsed Minutes": elapsed.total_seconds() / 60,
        })

time_df = pd.DataFrame(time_differences)

print("\nAUTHENTICATION-TO-SENSITIVE-ACTIVITY INTERVAL")
print(time_df.to_string(index=False))


AUTHENTICATION-TO-SENSITIVE-ACTIVITY INTERVAL
   User    Source IP Authentication Time    Sensitive Activity       Activity Time  Elapsed Minutes
buyer01 203.0.113.25 2026-10-09 09:04:00 Bank details modified 2026-10-09 09:10:00              6.0


7. Group events by source IP

In [7]:
ip_summary = (
    df.groupby("Source IP")
    .agg(
        Total_Events=("Event", "size"),
        Unique_Users=("User", "nunique"),
        Failed_Logins=(
            "Result",
            lambda s: (
                (s == "Failure")
                & df.loc[s.index, "Event"].eq("Login")
            ).sum(),
        ),
        Successful_Logins=(
            "Result",
            lambda s: (
                (s == "Success")
                & df.loc[s.index, "Event"].eq("Login")
            ).sum(),
        ),
        Sensitive_Changes=(
            "Event",
            lambda s: s.str.lower().isin(sensitive_keywords).sum(),
        ),
    )
    .reset_index()
)

print("\nEVENTS GROUPED BY SOURCE IP")
print(ip_summary.to_string(index=False))


EVENTS GROUPED BY SOURCE IP
   Source IP  Total_Events  Unique_Users  Failed_Logins  Successful_Logins  Sensitive_Changes
   10.0.0.12             1             1              0                  1                  0
203.0.113.25             5             1              2                  1                  1


8. Detect activity outside the user's normal working pattern

In [8]:
WORK_START_HOUR = 8
WORK_END_HOUR = 18

df["Hour"] = df["Timestamp"].dt.hour

df["Outside_Working_Hours"] = (
    (df["Hour"] < WORK_START_HOUR)
    | (df["Hour"] >= WORK_END_HOUR)
)

outside_hours = df[df["Outside_Working_Hours"]]

print("\nEVENTS OUTSIDE THE ASSUMED WORKING WINDOW")
print(
    outside_hours[
        ["Timestamp", "User", "Event", "Source IP"]
    ].to_string(index=False)
)

if outside_hours.empty:
    print("No events occurred outside the assumed working window.")


EVENTS OUTSIDE THE ASSUMED WORKING WINDOW
Empty DataFrame
Columns: [Timestamp, User, Event, Source IP]
Index: []
No events occurred outside the assumed working window.


9. Generate a chronological incident timeline

In [9]:
timeline = df[
    [
        "Timestamp",
        "User",
        "Event",
        "Source IP",
        "Result",
    ]
].copy()

timeline["Investigation Note"] = timeline.apply(
    lambda row: (
        "Successful login after two failures"
        if (
            row["Event"] == "Login"
            and row["Result"] == "Success"
            and row["Source IP"] == "203.0.113.25"
            and row["Timestamp"].strftime("%H:%M") == "09:04"
        )
        else "Sensitive supplier banking change"
        if row["Event"] == "Bank details modified"
        else "Supplier record accessed"
        if row["Event"] == "Supplier record viewed"
        else "Failed authentication attempt"
        if (
            row["Event"] == "Login"
            and row["Result"] == "Failure"
        )
        else "Initial successful authentication"
        if (
            row["Event"] == "Login"
            and row["Result"] == "Success"
        )
        else "Other event"
    ),
    axis=1,
)

timeline = timeline.sort_values("Timestamp").reset_index(drop=True)

print("\nCHRONOLOGICAL INCIDENT TIMELINE")
print(timeline.to_string(index=False))


CHRONOLOGICAL INCIDENT TIMELINE
          Timestamp    User                  Event    Source IP  Result                  Investigation Note
2026-10-09 09:00:00 buyer01                  Login    10.0.0.12 Success   Initial successful authentication
2026-10-09 09:02:00 buyer01                  Login 203.0.113.25 Failure       Failed authentication attempt
2026-10-09 09:03:00 buyer01                  Login 203.0.113.25 Failure       Failed authentication attempt
2026-10-09 09:04:00 buyer01                  Login 203.0.113.25 Success Successful login after two failures
2026-10-09 09:07:00 buyer01 Supplier record viewed 203.0.113.25 Success            Supplier record accessed
2026-10-09 09:10:00 buyer01  Bank details modified 203.0.113.25 Success   Sensitive supplier banking change


10. Calculate investigation duration

In [10]:
investigation_start = df["Timestamp"].min()
investigation_end = df["Timestamp"].max()

investigation_duration = (
    investigation_end - investigation_start
)

print(f"Investigation start: {investigation_start}")
print(f"Investigation end: {investigation_end}")
print(
    "Investigation duration: "
    f"{investigation_duration.total_seconds() / 60:.0f} minutes"
)

Investigation start: 2026-10-09 09:00:00
Investigation end: 2026-10-09 09:10:00
Investigation duration: 10 minutes


11. Consolidated security metrics

| Security metric                           | Result     | Interpretation                                                |
| ----------------------------------------- | ---------- | ------------------------------------------------------------- |
| Total events                              | 6          | Six recorded security events.                                 |
| Total login attempts                      | 4          | Four authentication attempts.                                 |
| Failed logins                             | 2          | Two unsuccessful authentication attempts.                     |
| Successful logins                         | 2          | Two successful authentication attempts.                       |
| Failed-login rate                         | 50%        | Half of login attempts failed.                                |
| Success after repeated failures           | Yes        | Success followed two failures from the same IP.               |
| Sensitive supplier changes                | 1          | One bank-details modification.                                |
| Authentication-to-sensitive-activity time | 6 minutes  | Modification occurred six minutes after the correlated login. |
| Source IPs                                | 2          | Two distinct source IPs were observed.                        |
| Events outside assumed working hours      | 0          | None fell outside 08:00–18:00.                                |
| Observed activity window                  | 10 minutes | First-to-last recorded event.                                 |
